In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Grundeinstellungen
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

FILE_PATH = r"uncleaned_aufschreibung_2024to2026.csv"
DELIMITER = ";"
TARGET_COL = "Störfall"

print(f"Lese unbereinigte Rohdaten ein von: {FILE_PATH}")
df_raw = pd.read_csv(FILE_PATH, sep=DELIMITER, low_memory=False)

# --- 1. Fokus auf DatumNEU und Intervall-Formatierung ---
df = df_raw.dropna(subset=['DatumNEU']).copy()
# Abschneiden möglicher Uhrzeiten im Datumsstring für einheitliches Format
df['DatumNEU'] = pd.to_datetime(df['DatumNEU'].str.split(' ').str[0], errors='coerce')
df = df.dropna(subset=['DatumNEU'])

# Zeit-Strings (z.B. '04:45:00') in saubere Minuten ab Mitternacht konvertieren
def time_to_minutes(t):
    t_str = str(t).strip()
    if pd.isna(t) or t_str == 'nan' or ':' not in t_str:
        return 0
    try:
        parts = t_str.split(':')
        return int(parts[0]) * 60 + int(parts[1])
    except:
        return 0

df['Zeit_von_min'] = df['Zeit von'].apply(time_to_minutes)
df['Zeit_bis_min'] = df['Zeit bis'].apply(time_to_minutes)

# --- 2. Target Sanity Check ---
# Störfall aufbauen anhand logischer Indikatoren (Ausfalldauern > 0)
for col in ['Dauer Anlagen-Ausfall', 'Dauer Anlagen-Ausfall intern']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors='coerce').fillna(0)

df[TARGET_COL] = ((df['Dauer Anlagen-Ausfall'] > 0) | (df['Dauer Anlagen-Ausfall intern'] > 0)).astype(int)

# --- 3. Ausschluss von Textrauschen / Typsicherung ---
num_cols = ['Dauer Arbeits-zeit', 'Anzahl MA', 'Anzahl/Std.', 'Sollzeit / Stück (Min)', 'Wochentag']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(',', '.'), errors='coerce').fillna(0)

# --- 4. Intervall-Deduplikation ---
# Verdichtung paralleler Aufschreibungen zu einzelnen Zeitschritten
agg_dict = {
    TARGET_COL: 'max',  # Wenn ein Störfall im Intervall auftrat, gilt das als 1
    'Dauer Arbeits-zeit': 'mean',
    'Anzahl MA': 'max',
    'Anzahl/Std.': 'mean',
    'Sollzeit / Stück (Min)': 'mean',
    'Wochentag': 'first'
}

df_agg = df.groupby(['DatumNEU', 'Zeit_von_min', 'Zeit_bis_min'], as_index=False).agg(agg_dict)

# --- 5. Sequentielle LSTM Feature-Engine ---
# Datumskomponenten
df_agg['Jahr'] = df_agg['DatumNEU'].dt.year
df_agg['Monat'] = df_agg['DatumNEU'].dt.month
df_agg['Tag'] = df_agg['DatumNEU'].dt.day
df_agg['Quartal'] = df_agg['DatumNEU'].dt.quarter

# Für die rollierenden Zeitfenster brauchen wir einen aufsteigenden sauberen Index
df_agg['datetime_start'] = df_agg['DatumNEU'] + pd.to_timedelta(df_agg['Zeit_von_min'], unit='m')
df_agg = df_agg.sort_values('datetime_start').set_index('datetime_start')

# Rolling Targets (7d & 30d Mean)
df_agg['Stoerfall_7d_mean'] = df_agg[TARGET_COL].rolling(window='7D').mean().fillna(0)
df_agg['Stoerfall_30d_mean'] = df_agg[TARGET_COL].rolling(window='30D').mean().fillna(0)

# Zeit seit dem letzten Störfall ausrechnen
df_agg = df_agg.reset_index()
failure_times = df_agg.loc[df_agg[TARGET_COL] == 1, 'datetime_start']

if not failure_times.empty:
    df_times = pd.DataFrame({'timestamp': df_agg['datetime_start']})
    df_failures = pd.DataFrame({'failure_time': failure_times})
    # Rückwärts-Merge, um den Zeitpunkt des letzten Vorfalls zu mappen
    df_merged = pd.merge_asof(df_times, df_failures, left_on='timestamp', right_on='failure_time', direction='backward')
    df_agg['Zeit_seit_letztem_Fehler_min'] = (df_merged['timestamp'] - df_merged['failure_time']).dt.total_seconds() / 60.0
else:
    df_agg['Zeit_seit_letztem_Fehler_min'] = 0.0

df_agg['Zeit_seit_letztem_Fehler_min'] = df_agg['Zeit_seit_letztem_Fehler_min'].fillna(0)

# Statische Kennzahl: Avg Time To Failure
avg_ttf = df_agg.loc[df_agg[TARGET_COL] == 1, 'Zeit_seit_letztem_Fehler_min'].mean()
df_agg['Avg_Time_To_Failure_min'] = avg_ttf if pd.notna(avg_ttf) else 0.0

# --- 6. Zusammenführung für Ausgabe (Exact Match Nomenklatur) ---
# Formatieren auf identische Labels der bisherigen Tabelle
df_agg = df_agg.rename(columns={
    'Anzahl/Std.': 'Anzahl/ Std.',
    'Sollzeit / Stück (Min)': 'Sollzeit/ Stück (Min)'
})

cols_for_stats = [
    'Wochentag', 'Jahr', 'Monat', 'Tag', 'Quartal',
    'Dauer Arbeits-zeit', 'Anzahl MA', 'Anzahl/ Std.', 'Sollzeit/ Stück (Min)',
    'Zeit_von_min', 'Zeit_bis_min', 'Zeit_seit_letztem_Fehler_min',
    'Stoerfall_7d_mean', 'Stoerfall_30d_mean', 'Avg_Time_To_Failure_min'
]

print("--- 1. STATISTIKEN NUMERISCHER FEATURES ---")
stats_df = pd.DataFrame({
    'count': df_agg[cols_for_stats].count(),
    'mean': df_agg[cols_for_stats].mean(),
    'median': df_agg[cols_for_stats].median(),
    'std': df_agg[cols_for_stats].std(),
    'min': df_agg[cols_for_stats].min(),
    'max': df_agg[cols_for_stats].max(),
    'IQR': df_agg[cols_for_stats].quantile(0.75) - df_agg[cols_for_stats].quantile(0.25),
    'skewness': df_agg[cols_for_stats].skew(),
    'kurtosis': df_agg[cols_for_stats].kurtosis()
})

display(stats_df.round(3))